# Cropy 작업자 분리 시험셋: 무 질병 전용 2단계 분류기 실험

448px 본 모델도 **무검은무늬병 212장 중 85장을 무노균병으로** 봅니다. 이 노트북은 무 질병 두 클래스만 가르는 전용 분류기를 **같은 작업자 분할의 학습 사진(무검은무늬병 267장 + 무노균병 124장)** 으로 학습하고, 시험 작업자 사진으로 다음을 비교합니다.

1. 본 모델(448px) 단독
2. 전용 분류기 단독 (무 질병 사진에서 두 클래스 중 선택)
3. 본 모델 + 전용 분류기, 백엔드 규칙(`recapture.apply_pair_specialist`): 본 모델 상위 2개가 {무검은무늬병, 무노균병}일 때만 두 클래스 합계 확률을 전용 분류기 비율로 재분배

PR #112의 공정 비교(촬영지·날짜 묶음 제외, 256px 본 모델)에서는 전용 분류기가 이득이 없었습니다(McNemar p=0.84). 이번에는 **작업자 분리 시험셋·448px 본 모델** 위에서 같은 질문을 다시 묻고, 작업자·촬영 시기·피해 정도별로 어디서 갈리는지 봅니다.

- 전용 분류기: ResNet50(ImageNet 가중치), 448px, 흐림 증강, 클래스 균형, lr 1e-4, 12 epoch. 사진이 400장이라 T4에서 15분 안팎입니다.
- 결과: `MyDrive/cropy_testset/pair_v1/` (`pair_mu.pth`, `test_report.json`, `predictions_pair_test.csv`)
- 데이터 준비(1번)는 이전 실험과 같아서 같은 세션이면 바로 넘어갑니다.

**GPU 런타임** 에서 위에서부터 차례로 실행하세요.

In [ ]:
from pathlib import Path
import csv, hashlib, json, re, shutil, subprocess, sys, zipfile
import numpy as np, torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
MY = Path('/content/drive/MyDrive')
TS = MY / 'cropy_testset'
RUN = TS / 'pair_v1'
WORK = Path('/content/cropy_testset')
DATA = WORK / 'dataset'
SPLIT = WORK / 'split'
EPOCHS, BATCH_SIZE, LR, IMG_SIZE = 12, 16, 1e-4, 448
EXPECTED_BASE_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_CODE_SHA256 = 'efe7034c6acd0544416e0cdb94c8fe748081921d73cf2fd1513fbc9b55e292a4'
SOURCES = {'base': MY / 'cropy_full_512/cropy_full_512.zip', 'code': MY / 'cropy_other10/training_code_other10.zip',
           'aihub': MY / 'cropy_other10/aihub_other_v1.zip', 'coco': MY / 'cropy_other10/coco_other_v1.zip',
           'server': MY / 'cropy_other10/cropy_other10_handoff_20261027/full512_resnet50_other10_v1.pth',
           'workers': TS / 'workers.json', 'trainer': TS / 'train_full_colab_convnext.py',
           'baseline': TS / 'baseline_v1/best_model.pth', 'main448': TS / 'hires448_v1/best_model.pth'}
for path in SOURCES.values():
    assert path.is_file(), f'Drive에 파일이 없습니다: {path}'
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)

## 1. 데이터 준비와 작업자 단위 재분할
Drive에서 로컬로 복사해 풀고, 모든 사진을 작업자 배정표대로 `split/{train,val,test}/<클래스>/`에 옮깁니다. AI Hub 9클래스 사진 수가 설계와 정확히 같은지 확인합니다.

In [ ]:
EXPECT = {"train": {"0": 589, "1": 528, "2": 267, "3": 124, "4": 574, "5": 310, "6": 5932, "7": 6726, "8": 5724}, "val": {"0": 134, "1": 166, "2": 59, "3": 49, "4": 107, "5": 58, "6": 1440, "7": 2023, "8": 1688}, "test": {"0": 361, "1": 330, "2": 212, "3": 78, "4": 230, "5": 125, "6": 2313, "7": 3349, "8": 3087}}

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack(src, dest, expected=None):
    local = WORK / src.name
    if not local.exists():
        shutil.copyfile(src, local)
    if expected:
        assert digest(local) == expected, f'해시 불일치: {src.name}'
    with zipfile.ZipFile(local) as z:
        for item in z.infolist():
            assert (dest / item.filename).resolve().is_relative_to(dest.resolve()), '잘못된 압축 경로'
        z.extractall(dest)

unpack(SOURCES['code'], WORK / 'code', EXPECTED_CODE_SHA256)
ASSIGN = json.loads(SOURCES['workers'].read_text(encoding='utf-8'))['assign']
GROUP = re.compile(r'(?:^|_)([0-9A-Za-z]+)_(20\d{6})(?=_|\.|$)')
def split_of(path):
    m = GROUP.search(path.name)
    if m:
        return ASSIGN[m.group(1)] if m.group(1) in ASSIGN else 'train'
    if path.parent.name.startswith('9_'):  # COCO 조각: 내용 해시로 70/10/20
        b = int(digest(path)[:8], 16) % 10
        return 'train' if b < 7 else 'val' if b == 7 else 'test'
    return 'train'

if not (WORK / 'split_done').exists():
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.rmtree(SPLIT, ignore_errors=True)
    unpack(SOURCES['base'], WORK, EXPECTED_BASE_SHA256)
    (DATA / 'manifest.csv').rename(DATA / 'manifest_9class.csv')
    unpack(SOURCES['aihub'], WORK)
    unpack(SOURCES['coco'], WORK)
    rows = []
    for src_split in ('train', 'val'):
        for path in sorted((DATA / 'images' / src_split).glob('*/*')):
            s = split_of(path)
            dest = SPLIT / s / path.parent.name / path.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), dest)
            m = GROUP.search(path.name)
            rows.append([s, path.parent.name, path.name, m.group(1) if m else '', src_split])
    with (RUN / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['split', 'class', 'file', 'worker', 'official_split'])
        w.writerows(rows)
    (WORK / 'split_done').touch()

counts = {s: {c.name: len(list(c.iterdir())) for c in sorted((SPLIT / s).iterdir())} for s in ('train', 'val', 'test')}
for s, cs in counts.items():
    print(s, sum(cs.values()), cs)
for s, cs in EXPECT.items():
    for c, n in cs.items():
        got = counts[s][[k for k in counts[s] if k.startswith(c + '_')][0]]
        assert got == n, f'{s}/{c}: {got} != 설계 {n}'
print('AI Hub 9클래스 사진 수가 설계와 일치합니다.')
fingerprint = ''.join(f'{name}: {digest(WORK / SOURCES[name].name)}\n' for name in ('base', 'aihub', 'coco'))
fingerprint += f"workers: {digest(SOURCES['workers'])}\n"
(SPLIT.parent / 'manifest.csv').write_text(fingerprint, encoding='utf-8')  # 학습 스크립트는 data-dir 상위의 manifest.csv를 읽는다
from torchvision.datasets import ImageFolder
train_ds, val_ds = ImageFolder(SPLIT / 'train'), ImageFolder(SPLIT / 'val')
assert train_ds.classes == val_ds.classes and len(train_ds.classes) == 10, train_ds.classes
EXPECT_IMAGES = (len(train_ds), len(val_ds))
(RUN / 'split_counts.json').write_text(json.dumps(counts, ensure_ascii=False, indent=1), encoding='utf-8')
print('학습/검증:', EXPECT_IMAGES)

## 2. 전용 분류기 학습 (무검은무늬병 vs 무노균병)
학습 폴더는 작업자 분할의 `train`/`val`에서 두 클래스만 링크해 만듭니다. 체크포인트의 `classes`가 `2_…`, `3_…` 두 개라 백엔드 `pair_mu.pth` 형식과 같습니다.

In [ ]:
PAIR = WORK / 'pair'
shutil.rmtree(PAIR, ignore_errors=True)
for s in ('train', 'val'):
    for c in sorted((SPLIT / s).iterdir()):
        if c.name.startswith(('2_', '3_')):
            (PAIR / 'images' / s / c.name).parent.mkdir(parents=True, exist_ok=True)
            (PAIR / 'images' / s / c.name).symlink_to(c, target_is_directory=True)
shutil.copyfile(SPLIT.parent / 'manifest.csv', PAIR / 'manifest.csv')  # 학습 스크립트는 data-dir 상위의 manifest.csv를 읽는다
from torchvision.datasets import ImageFolder
pair_train, pair_val = ImageFolder(PAIR / 'images/train'), ImageFolder(PAIR / 'images/val')
assert pair_train.classes == pair_val.classes and len(pair_train.classes) == 2, pair_train.classes
PAIR_IMAGES = (len(pair_train), len(pair_val))
print('전용 분류기 학습/검증:', PAIR_IMAGES, pair_train.classes)
shutil.copyfile(SOURCES['trainer'], WORK / 'code/train_full_colab.py')
command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
    '--data-dir', str(PAIR / 'images'), '--output-dir', str(RUN / 'main'),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
    '--num-workers', '2', '--img-size', str(IMG_SIZE), '--arch', 'resnet50', '--blur-aug',
    '--expect-images', str(PAIR_IMAGES[0]), str(PAIR_IMAGES[1])]
if (RUN / 'main/last_checkpoint.pth').exists():
    command.append('--resume')
print('이어 학습:', '--resume' in command)
subprocess.run(command, check=True)
shutil.copyfile(RUN / 'main/best_model.pth', RUN / 'pair_mu.pth')

## 3. 시험 작업자 사진으로 비교
본 모델(448px)과 전용 분류기를 시험셋 전체에 돌리고, 무 질병 290장에서 세 방식을 비교합니다. 작업자·촬영 시기·피해 정도(파일명 risk 토큰)별 정답 수와 McNemar 검정을 함께 냅니다.

In [ ]:
from PIL import Image
from torchvision import transforms
from math import comb

def load(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    sys.path.insert(0, str(WORK / 'code'))
    from train_full_colab import build_model
    model = build_model(ckpt.get('arch', 'resnet50'), ckpt['num_classes'], pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    return model.cuda().eval(), ckpt

def predict(model, size, files):
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    out = []
    for i in range(0, len(files), 64):
        batch = files[i:i + 64]
        with torch.no_grad():
            out.append(model(torch.stack([tf(Image.open(p).convert('RGB')) for p in batch]).cuda()).softmax(1).cpu().numpy())
    return np.concatenate(out)

files = sorted((SPLIT / 'test').glob('*/*'))
y = np.array([int(p.parent.name.split('_')[0]) for p in files])
main, mck = load(SOURCES['main448'])
main_ids = [int(c.split('_', 1)[0]) for c in mck['classes']]
P = predict(main, mck.get('img_size', 256), files)   # (N, 10)
del main
spec, sck = load(RUN / 'pair_mu.pth')
spec_ids = [int(c.split('_', 1)[0]) for c in sck['classes']]
assert sorted(spec_ids) == [2, 3], spec_ids
S = predict(spec, sck.get('img_size', 256), files)   # (N, 2)
del spec
a, b = spec_ids.index(2), spec_ids.index(3)
i2, i3 = main_ids.index(2), main_ids.index(3)

# 방식 1: 본 모델
pred_main = np.array(main_ids)[P.argmax(1)]
# 방식 3: 백엔드 규칙 — 상위 2개가 {2,3}일 때만 두 클래스 합계를 전용 비율로 재분배
F = P.copy()
top2 = np.argsort(-P, axis=1)[:, :2]
gate = np.array([set(main_ids[j] for j in t) == {2, 3} for t in top2])
comb_mass = P[gate, i2] + P[gate, i3]
ratio = S[gate, a] / np.maximum(S[gate, a] + S[gate, b], 1e-9)
F[gate, i2] = comb_mass * ratio
F[gate, i3] = comb_mass * (1 - ratio)
pred_fused = np.array(main_ids)[F.argmax(1)]
# 방식 2: 전용 단독 (무 질병 사진에만 의미 있음)
pred_spec = np.where(S[:, a] >= S[:, b], 2, 3)

rad = np.isin(y, [2, 3])
def tok(p, k):
    t = p.name.split('_')
    return {'worker': t[8], 'date': t[9], 'risk': t[7]}[k] if len(t) > 9 else ''
def summarize(pred, mask):
    yy, pp = y[mask], pred[mask]
    r2 = float((pp[yy == 2] == 2).mean()); r3 = float((pp[yy == 3] == 3).mean())
    return dict(n=int(mask.sum()), accuracy=round(float((pp == yy).mean()), 4), recall_2=round(r2, 4), recall_3=round(r3, 4),
                balanced=round((r2 + r3) / 2, 4))
def breakdown(pred, key):
    out = {}
    for p, yy, pp in zip(files, y, pred):
        if yy in (2, 3):
            k = f'{yy}/{tok(p, key)}'
            out.setdefault(k, [0, 0]); out[k][0] += int(pp == yy); out[k][1] += 1
    return {k: f'{v[0]}/{v[1]}' for k, v in sorted(out.items())}
def mcnemar(p1, p2, mask):
    c1 = (p1[mask] == y[mask]); c2 = (p2[mask] == y[mask])
    n01, n10 = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    n = n01 + n10; k = min(n01, n10)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    return dict(only_first=n01, only_second=n10, p=round(p, 4))

report = dict(pair_epoch=int(sck['epoch']), pair_train_images=PAIR_IMAGES, gate_applied=int(gate.sum()),
              gate_applied_on_radish_disease=int((gate & rad).sum()))
for name, pred in (('main448', pred_main), ('specialist', pred_spec), ('fused_backend_rule', pred_fused)):
    report[name] = dict(radish_disease=summarize(pred, rad),
                        by_worker=breakdown(pred, 'worker'), by_risk=breakdown(pred, 'risk'), by_date=breakdown(pred, 'date'))
    if name != 'specialist':
        report[name]['all_test'] = dict(n=int(len(y)), accuracy=round(float((pred == y).mean()), 4))
report['mcnemar_main_vs_specialist'] = mcnemar(pred_main, pred_spec, rad)
report['mcnemar_main_vs_fused'] = mcnemar(pred_main, pred_fused, rad)
# 전용 분류기가 무 질병이 아닌 사진에 끼친 영향: 게이트가 열린 비무질병 사진 수와 그중 바뀐 예측
nonrad_gate = gate & ~rad
report['fused_changed_nonradish'] = int((pred_fused[nonrad_gate] != pred_main[nonrad_gate]).sum())
with (RUN / 'predictions_pair_test.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['file', 'label', 'worker', 'main_pred', 'spec_pred', 'fused_pred', 'main_p2', 'main_p3', 'spec_p2', 'spec_p3'])
    for p, yy, pm, ps, pf, pr, sr in zip(files, y, pred_main, pred_spec, pred_fused, P, S):
        w.writerow([p.name, int(yy), tok(p, 'worker'), int(pm), int(ps), int(pf), round(float(pr[i2]), 5), round(float(pr[i3]), 5), round(float(sr[a]), 5), round(float(sr[b]), 5)])
(RUN / 'test_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
for name in ('main448', 'specialist', 'fused_backend_rule'):
    r = report[name]['radish_disease']
    print(f"{name:20s} 무 질병 290장: 정확도 {r['accuracy']:.1%}, 검은무늬 {r['recall_2']:.1%}, 노균 {r['recall_3']:.1%}, 균형 {r['balanced']:.1%}")
print('McNemar 본 vs 전용', report['mcnemar_main_vs_specialist'], '| 본 vs 융합', report['mcnemar_main_vs_fused'])
print('저장:', RUN / 'test_report.json')